In [ ]:
!pip install tqdm
!pip install spacy
!python -m spacy download en_core_web_sm

In [2]:
import pandas as pd
import os
import re
import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm
import ast
from utils_challenge1 import extraire_contexte_mot, match_tableaux, reconstruire_embeddings, retour_dataset_HVD, extraire_contexte_mot_df_HVD
import pickle
import spacy
import matplotlib.pyplot as plt

from pathlib import Path
import json
import pyarrow

nlp = spacy.load("en_core_web_sm")
stop_words_spacy = nlp.Defaults.stop_words

In [ ]:
matrice_A = np.fromfile("6B.300d.bin", dtype=np.float32).reshape(300, 300)

In [ ]:
glove = {}
with open("glove.6B.300d.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        glove[word] = vector

In [ ]:
#df_imm=extraire_contexte_mot("immigration",rayon=5,supprimer_stopwords=True)
#df_imm=match_tableaux(df_imm)
#df_fed=extraire_contexte_mot("federal_reserve",rayon=5, multi_mot = True, supprimer_stopwords=True)
#df_fed=match_tableaux(df_fed)
#df_mon=extraire_contexte_mot("monetary_policy",rayon=5, multi_mot = True, supprimer_stopwords=True)
#df_mon=match_tableaux(df_mon)

In [ ]:
df_imm=pd.read_csv("immigration5stopEXCLU.csv")
df_imm=reconstruire_embeddings(df_imm, glove, matrice_A)

In [ ]:
df_fed=pd.read_csv("federal_reserve5stopEXCLU.csv")
df_fed=reconstruire_embeddings(df_fed,glove,matrice_A)

In [ ]:
df_mon=pd.read_csv("monetary_policy5stopEXCLU.csv")
df_mon=reconstruire_embeddings(df_mon,glove,matrice_A)

# Deuxième Dataset plus propre - DATASET HARVARD

In [ ]:
#df=retour_dataset_HVD("datasetHarvard")
#df_HVD_imm=extraire_contexte_mot_HVD(df,"immigration",rayon=5,supprimer_stopwords=True)
#df_HVD_imm=match_tableaux(df_HVD_imm)
#df_HVD_fed=extraire_contexte_mot_HVD(df,"federal_reserve",rayon=5, multi_mot = True, supprimer_stopwords=True)
#df_HVD_fed=match_tableaux(df_HVD_fed)
#df_HVD_mon=extraire_contexte_mot_HVD(df,"monetary_policy",rayon=5, multi_mot = True, supprimer_stopwords=True)
#df_HVD_mon=match_tableaux(df_HVD_mon)

In [ ]:
df_HVD_imm=pd.read_csv("HVDimmigration5stopEXCLU.csv")
df_HVD_imm=reconstruire_embeddings(df_HVD_imm, glove, matrice_A)

In [ ]:
df_HVD_fed=pd.read_csv("HVDfederal_reserve5stopEXCLU.csv")
df_HVD_fed=reconstruire_embeddings(df_HVD_fed,glove,matrice_A)

In [ ]:
df_HVD_mon = pd.read_csv('HVDmonetary_policy5stopEXCLU.csv')
df_HVD_mon=reconstruire_embeddings(df_HVD_mon, glove, matrice_A)

# Regression linéaire multivariée

In [ ]:
def fit_embedding_regression(df: pd.DataFrame, embed_col='embedding', party_col='party'):
    """
    OLS multivariée: Y = α·1 + β·1(Républicain) + ε
    - Filtre sur party ∈ {'R','D'}
    - Garde uniquement les lignes avec embeddings bien formés (même dimension)
    - Retourne α (intercept), β (vecteur d'effet R), T_obs = ||β||₂, et les moyennes de groupe
    """
    # 1) Filtre R/D
    dfin = df[df[party_col].isin(['R', 'D'])].copy()
    if dfin.empty:
        raise ValueError("Aucune ligne avec party ∈ {'R','D'} après filtrage.")

    # 2) Assurer une dimension d'embedding cohérente + drop NA
    dfin = dfin[dfin[embed_col].notna()].copy()
    # Convertit en arrays et filtre si longueurs incohérentes
    emb_as_array = []
    dims = None
    keep_idx = []
    for i, emb in enumerate(dfin[embed_col].tolist()):
        # tolère list/np.array
        arr = np.asarray(emb, dtype=float)
        if arr.ndim != 1:
            continue
        if dims is None:
            dims = arr.shape[0]
            if dims == 0:
                raise ValueError("Embeddings vides.")
        if arr.shape[0] != dims or np.any(~np.isfinite(arr)):
            continue
        emb_as_array.append(arr)
        keep_idx.append(True)
    if not emb_as_array:
        raise ValueError("Aucun embedding valide trouvé (dimensions incohérentes ou NaN).")

    dfin = dfin.iloc[np.where(keep_idx)[0]].copy()
    Y = np.vstack(emb_as_array)                     # (n, d)

    # 3) Matrice de design X = [constante, 1(R)]
    R_ind = (dfin[party_col] == 'R').astype(int).to_numpy().reshape(-1, 1)
    X = np.hstack([np.ones((len(dfin), 1)), R_ind])  # (n, 2)

    # 4) OLS multivariée en une passe (lstsq)
    #    B a la forme (2, d) : 1ère ligne = α, 2ème = β
    B, *_ = np.linalg.lstsq(X, Y, rcond=None)
    alpha = B[0, :]                 # embedding moyen pour le groupe de référence (D)
    beta  = B[1, :]                 # déplacement sémantique R vs D
    T_obs = float(np.linalg.norm(beta, ord=2))

    # 5) Moyennes de groupe (utile pour l’interprétation)
    mean_D = alpha
    mean_R = alpha + beta

    # (Optionnel) R^2 multivarié global (variance expliquée moyenne par dimension)
    Yhat = X @ B
    ss_res = np.sum((Y - Yhat) ** 2, axis=0)
    ss_tot = np.sum((Y - Y.mean(axis=0)) ** 2, axis=0)
    R2_per_dim = 1 - ss_res / np.maximum(ss_tot, 1e-12)
    R2_mean = float(np.mean(R2_per_dim))

    return {
        "alpha": alpha,          # np.array (d,)
        "beta": beta,            # np.array (d,)
        "T_obs_norm_beta": T_obs,
        "mean_D": mean_D,        # np.array (d,)
        "mean_R": mean_R,        # np.array (d,)
        "R2_mean": R2_mean,
        "n_used": Y.shape[0],
        "d": Y.shape[1],
    }

In [ ]:
# --- Exemple d’usage ---
results_imm = fit_embedding_regression(df_imm, embed_col='embedding', party_col='party')
print("||β||₂ =", results_imm["T_obs_norm_beta"], "| n =", results_imm["n_used"], " | d =", results_imm["d"])

In [ ]:
# --- Exemple d’usage ---
results_HVD_imm = fit_embedding_regression(df_HVD_imm, embed_col='embedding', party_col='party')
print("||β||₂ =", results_HVD_imm["T_obs_norm_beta"], "| n =", results_HVD_imm["n_used"], " | d =", results_HVD_imm["d"])

In [ ]:
results_fed = fit_embedding_regression(df_fed, embed_col='embedding', party_col='party')
print("||β||₂ =", results_fed["T_obs_norm_beta"], "| n =", results_fed["n_used"], " | d =", results_fed["d"])

In [ ]:
results_HVD_fed = fit_embedding_regression(df_HVD_fed, embed_col='embedding', party_col='party')
print("||β||₂ =", results_HVD_fed["T_obs_norm_beta"], "| n =", results_HVD_fed["n_used"], " | d =", results_HVD_fed["d"])

In [ ]:
results_mon = fit_embedding_regression(df_mon, embed_col='embedding', party_col='party')
print("||β||₂ =", results_mon["T_obs_norm_beta"], "| n =", results_mon["n_used"], " | d =", results_mon["d"])

In [ ]:
results_HVD_mon = fit_embedding_regression(df_HVD_mon, embed_col='embedding', party_col='party')
print("||β||₂ =", results_HVD_mon["T_obs_norm_beta"], "| n =", results_HVD_mon["n_used"], " | d =", results_HVD_mon["d"])

# Test par permutation

In [ ]:
# --- 1) Utilitaire: parser les embeddings proprement (comme dans ta fonction) ---
def _parse_embeddings(df, embed_col='embedding', party_col='party'):
    dfin = df[df[party_col].isin(['R','D'])].copy()
    if dfin.empty:
        raise ValueError("Aucune ligne avec party ∈ {'R','D'} après filtrage.")
    dfin = dfin[dfin[embed_col].notna()].copy()

    emb_arrs = []
    keep_mask = []
    dims = None
    for emb in dfin[embed_col].tolist():
        arr = np.asarray(emb, dtype=float)
        if arr.ndim != 1:
            keep_mask.append(False); continue
        if dims is None:
            dims = arr.shape[0]
            if dims == 0:
                raise ValueError("Embeddings vides.")
        if arr.shape[0] != dims or np.any(~np.isfinite(arr)):
            keep_mask.append(False); continue
        emb_arrs.append(arr)
        keep_mask.append(True)

    if not emb_arrs:
        raise ValueError("Aucun embedding valide (dimensions incohérentes ou NaN).")

    keep_mask = np.array(keep_mask, dtype=bool)
    dfin = dfin.loc[keep_mask].reset_index(drop=True)   # <<<<<< important
    Y = np.vstack(emb_arrs)                              # (n, d)
    R = (dfin[party_col] == 'R').astype(int).to_numpy()  # (n,)
    return dfin, Y, R

# --- 2) Norme de beta sans refitter l’OLS: beta = mean(Y|R=1) - mean(Y|R=0) ---
def _beta_and_T_from_R(Y, R):
    # sécurité : 2 groupes non vides
    if R.sum() == 0 or R.sum() == len(R):
        return None, np.nan
    mean_R = Y[R == 1].mean(axis=0)
    mean_D = Y[R == 0].mean(axis=0)
    beta = mean_R - mean_D
    T = float(np.linalg.norm(beta, ord=2))
    return beta, T

# --- 3) Générateur de permutations (par lignes, par grappe, et/ou stratifiées) ---
def _permute_R(dfin, R, group_col=None, strata_cols=None, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    R_perm = np.empty_like(R)

    if strata_cols is None or len(strata_cols) == 0:
        strata_key = pd.Series(0, index=dfin.index)
    else:
        strata_key = dfin[strata_cols].astype(str).agg('||'.join, axis=1)

    if group_col is None:
        # permutation simple au niveau des lignes, par strate
        for _, idx in strata_key.groupby(strata_key).groups.items():
            idx = np.fromiter(idx, dtype=int)        # idx est déjà 0..n-1
            R_perm[idx] = rng.permutation(R[idx])
    else:
        # permutation au niveau des groupes, par strate
        groups = dfin[group_col].to_numpy()
        for _, idx in strata_key.groupby(strata_key).groups.items():
            idx = np.fromiter(idx, dtype=int)
            g_s = groups[idx]
            df_s = pd.DataFrame({'g': g_s, 'R': R[idx]})
            lab_by_g = df_s.groupby('g')['R'].mean().round().astype(int)
            permuted_labels = lab_by_g.sample(
                frac=1.0,
                random_state=int(rng.integers(0, 2**32-1))
            )
            map_lab = permuted_labels.to_dict()
            R_perm[idx] = [map_lab[gg] for gg in g_s]

    return R_perm

# --- 4) Test par permutation ---
def permutation_pvalue_for_df(df, embed_col='embedding', party_col='party',
                              B=1000, random_state=0, group_col=None, strata_cols=None):
    """
    Calcule T_obs = ||beta||_2 et la p-value (>=) par permutation des étiquettes de parti.
    - group_col: ex. 'speakerid' pour permuter par grappe
    - strata_cols: ex. ['chamber','year'] pour permuter à l'intérieur des strates
    """
    rng = np.random.default_rng(random_state)

    dfin, Y, R = _parse_embeddings(df, embed_col=embed_col, party_col=party_col)

    # T observé
    beta_obs, T_obs = _beta_and_T_from_R(Y, R)
    if np.isnan(T_obs):
        raise ValueError("Impossible de calculer T_obs: un des groupes est vide.")

    # Boucle de permutations
    count_ge = 0
    T_perm = np.empty(B, dtype=float)
    for b in range(B):
        Rb = _permute_R(dfin, R, group_col=group_col, strata_cols=strata_cols, rng=rng)
        _, Tb = _beta_and_T_from_R(Y, Rb)
        T_perm[b] = Tb
        if Tb >= T_obs:
            count_ge += 1

    pval = (1 + count_ge) / (B + 1)
    return {
        'T_obs': T_obs,
        'p_value': pval,
        'T_perm': T_perm,     # utile si tu veux tracer la distribution
        'n_used': Y.shape[0],
        'd': Y.shape[1],
        'B': B
    }

In [ ]:
# --- Exemple d'usage minimal (sans cluster ni strates) ---
res_imm = permutation_pvalue_for_df(df_imm, embed_col='embedding', party_col='party', B=2000, random_state=42)
print("T_obs =", res_imm['T_obs'], " | p-value =", res_imm['p_value'])

In [ ]:
res_fed = permutation_pvalue_for_df(df_fed, embed_col='embedding', party_col='party', B=2000, random_state=42)
print("T_obs =", res_fed['T_obs'], " | p-value =", res_fed['p_value'])

In [ ]:
res_mon = permutation_pvalue_for_df(df_mon, embed_col='embedding', party_col='party', B=2000, random_state=42)
print("T_obs =", res_mon['T_obs'], " | p-value =", res_mon['p_value'])

# Evolution de la norme de Beta

In [ ]:
def fit_embedding_regression_by_year(
    df: pd.DataFrame,
    embed_col: str = 'embedding',
    party_col: str = 'party',
    date_col: str = 'date'
) -> pd.DataFrame:
    """
    Calcule, pour chaque année, la norme de β issue de la régression
    Y = α + β·1(R) + ε
    où Y est l'embedding contextuel (vectoriel).

    Sortie : DataFrame avec colonnes :
        - 'year'       : année (int)
        - 'beta_norm'  : ||β||₂ (float) pour l'année
        - 'n_R'        : nombre d'occurrences républicaines
        - 'n_D'        : nombre d'occurrences démocrates
    """
    # 1) Filtre R/D + embeddings non nuls
    dfin = df[df[party_col].isin(['R', 'D'])].copy()
    dfin = dfin[dfin[embed_col].notna()].copy()
    if dfin.empty:
        return pd.DataFrame(columns=['year', 'beta_norm', 'n_R', 'n_D'])

    # 2) Assure un champ 'year'
    if not np.issubdtype(dfin[date_col].dtype, np.datetime64):
        dfin[date_col] = pd.to_datetime(dfin[date_col], errors='coerce')
    dfin = dfin.dropna(subset=[date_col]).copy()
    if dfin.empty:
        return pd.DataFrame(columns=['year', 'beta_norm', 'n_R', 'n_D'])
    dfin['year'] = dfin[date_col].dt.year

    # 3) Convertit les embeddings en arrays et vérifie dimension
    d = None
    keep_idx = []
    for emb in dfin[embed_col].tolist():
        arr = np.asarray(emb, dtype=float)
        if arr.ndim != 1 or np.any(~np.isfinite(arr)):
            keep_idx.append(False)
            continue
        if d is None:
            d = arr.shape[0]
            if d == 0:
                keep_idx.append(False)
                continue
        if arr.shape[0] != d:
            keep_idx.append(False)
            continue
        keep_idx.append(True)

    dfin = dfin.iloc[np.where(keep_idx)[0]].copy()
    if dfin.empty:
        return pd.DataFrame(columns=['year', 'beta_norm', 'n_R', 'n_D'])

    # 4) Regroupe par année et calcule ||β||₂ + nb occurrences
    rows = []
    for yr, g in dfin.groupby('year', sort=True):
        g_R = g[g[party_col] == 'R'][embed_col].tolist()
        g_D = g[g[party_col] == 'D'][embed_col].tolist()
        n_R, n_D = len(g_R), len(g_D)
        if n_R == 0 or n_D == 0:
            continue

        mean_R = np.mean(np.vstack([np.asarray(x, dtype=float) for x in g_R]), axis=0)
        mean_D = np.mean(np.vstack([np.asarray(x, dtype=float) for x in g_D]), axis=0)
        beta = mean_R - mean_D
        beta_norm = float(np.linalg.norm(beta))

        rows.append((int(yr), beta_norm, n_R, n_D))

    out = pd.DataFrame(rows, columns=['year', 'beta_norm', 'n_R', 'n_D']).sort_values('year').reset_index(drop=True)
    return out

def plot_beta_evolution(df_beta_year, mot):
    """
    Trace l'évolution de ||β||₂ au fil du temps pour un mot donné.
    """
    plt.figure(figsize=(10,5))
    plt.plot(df_beta_year['year'], df_beta_year['beta_norm'], marker='o', linestyle='-')
    
    plt.title(f"Évolution temporelle de ||β||₂ pour {mot}")
    plt.xlabel("Année")
    plt.ylabel("Norme de β (||β||₂)")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
df_beta_imm = fit_embedding_regression_by_year(df_imm)
plot_beta_evolution(df_beta_imm, "immigration")

In [ ]:
df_beta_HVD_imm = fit_embedding_regression_by_year(df_HVD_imm)
plot_beta_evolution(df_beta_HVD_imm, "immigration")

In [ ]:
df_beta_fed = fit_embedding_regression_by_year(df_fed)
plot_beta_evolution(df_beta_fed, "Federal Reserve")

In [ ]:
df_beta_HVD_fed = fit_embedding_regression_by_year(df_HVD_fed)
plot_beta_evolution(df_beta_HVD_fed, "Federal Reserve")

In [ ]:
df_beta_mon = fit_embedding_regression_by_year(df_mon)
plot_beta_evolution(df_beta_mon, "Monetary Policy")

In [ ]:
df_beta_HVD_mon = fit_embedding_regression_by_year(df_HVD_mon)
plot_beta_evolution(df_beta_HVD_mon, "Monetary Policy")

# Corrélation avec indices boursiers

In [ ]:
import yfinance as yf

# Télécharger données S&P 500
sp500 = yf.download("^GSPC", start="1995-01-01", end="2025-01-01")
sp500["year"] = sp500.index.year
sp500.columns = [col[0] if isinstance(col, tuple) else col for col in sp500.columns]

# Rendement annuel basé sur le dernier cours de l'année
annual_close = sp500.groupby("year")["Close"].last()
sp500_return = annual_close.pct_change()

# Volatilité annuelle (écart-type des rendements quotidiens)
daily_ret = sp500["Close"].pct_change()
sp500_vol = daily_ret.groupby(sp500["year"]).std()

# --- Volumes ---
annual_vol_mean = sp500.groupby("year")["Volume"].mean()
annual_vol_sum  = sp500.groupby("year")["Volume"].sum()
# croissance du volume (différence du log ~ variation en %)
sp500_volume_growth = np.log(annual_vol_mean).diff()

# Construire un DataFrame avec S&P 500 agrégé
df_sp500 = pd.DataFrame({
    "sp500_close": annual_close,
    "sp500_return": sp500_return,
    "sp500_volatility": sp500_vol,
    "sp500_volume_mean": annual_vol_mean,
    "sp500_volume_sum": annual_vol_sum,
    "sp500_volume_growth": sp500_volume_growth,
}).reset_index()

# Fusionner avec tes DataFrames de polarisation
df_all = df_sp500.merge(
    df_beta_HVD_imm[["year", "beta_norm"]].rename(columns={"beta_norm": "beta_immigration"}),
    on="year", how="left"
).merge(
    df_beta_HVD_fed[["year", "beta_norm"]].rename(columns={"beta_norm": "beta_fed"}),
    on="year", how="left"
).merge(
    df_beta_HVD_mon[["year", "beta_norm"]].rename(columns={"beta_norm": "beta_monetary"}),
    on="year", how="left"
)

In [ ]:
df_all

lien entre polarisation et rendement ou volatilité du s&p ?

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12,6))

# Polarisation (traits pleins)
ax1.plot(df_all["year"], df_all["beta_immigration"], label="Immigration", color="tab:green")
ax1.plot(df_all["year"], df_all["beta_fed"], label="Federal Reserve", color="tab:blue")
ax1.plot(df_all["year"], df_all["beta_monetary"], label="Monetary Policy", color="tab:red")

ax1.set_xlabel("Année")
ax1.set_ylabel("Polarisation (‖β‖₂)")

# Axe secondaire : S&P 500 return en pointillé
ax2 = ax1.twinx()
ax2.plot(df_all["year"], df_all["sp500_return"], label="S&P 500 return",
         color="black", linestyle="--", linewidth=2)
ax2.set_ylabel("S&P 500 annual return")

# Récupérer uniquement les handles/labels des deux axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
plt.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Polarisation thématique vs Rendements S&P 500")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12,6))

# Polarisation (traits pleins)
ax1.plot(df_all["year"], df_all["beta_fed"], label="Federal Reserve", color="tab:blue")
ax1.plot(df_all["year"], df_all["beta_monetary"], label="Monetary Policy", color="tab:red")

ax1.set_xlabel("Année")
ax1.set_ylabel("Polarisation (‖β‖₂)")

# Axe secondaire : volatilité S&P 500 (pointillé)
ax2 = ax1.twinx()
ax2.plot(df_all["year"], df_all["sp500_volatility"], label="S&P 500 volatility",
         color="black", linestyle="--")

ax2.set_ylabel("Volatilité annuelle")

# Récupérer les handles/labels des deux axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Polarisation économique vs Volatilité du S&P 500")
plt.show()

In [ ]:
# Extraire juste les colonnes utiles
subset = df_all[["beta_fed", "beta_monetary", "sp500_volatility"]]

# Corrélations de Pearson (linéaires)
print("Corrélations de Pearson :")
print(subset.corr(method="pearson"))

# Corrélations de Spearman (rang, robustes aux valeurs extrêmes)
print("\nCorrélations de Spearman :")
print(subset.corr(method="spearman"))


In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

# Préparer les données : rendements et polarisation
df_gc_ret = df_all[["sp500_return", "beta_monetary"]].dropna()

print("Granger causality test : Polarisation Monetary Policy -> S&P500 return")
grangercausalitytests(df_gc_ret[["sp500_return", "beta_monetary"]], maxlag=3)

print("\nGranger causality test : S&P500 return -> Polarisation Monetary Policy")
grangercausalitytests(df_gc_ret[["beta_monetary", "sp500_return"]], maxlag=3)

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

# Préparer les données : supprimer les NaN et garder les deux colonnes
df_gc2 = df_all[["beta_monetary", "sp500_volatility"]].dropna()

# Test de Granger : est-ce que volume -> polarisation ?
# maxlag = nombre d'années de retard testées (ex. 3 ans)
print("Granger causality test : Volatilité -> Monetary Policy")
grangercausalitytests(df_gc2[["beta_monetary", "sp500_volatility"]], maxlag=3)

# Inverse : polarisation -> volume
print("\nGranger causality test : Monetary Policy -> Volatilité")
grangercausalitytests(df_gc2[["sp500_volatility", "beta_monetary"]], maxlag=3)


lien entre polarisation et volume ou croissance du volume du s&p ?

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12,6))

# Polarisation Fed et Monetary Policy
ax1.plot(df_all["year"], df_all["beta_fed"], label="Federal Reserve", color="tab:blue")
ax1.plot(df_all["year"], df_all["beta_monetary"], label="Monetary Policy", color="tab:red")
ax1.set_xlabel("Année")
ax1.set_ylabel("Polarisation (‖β‖₂)")

# Axe secondaire : volume moyen (échelle log pour lisibilité)
ax2 = ax1.twinx()
ax2.plot(df_all["year"], df_all["sp500_volume_mean"], label="S&P 500 volume (moyen)", 
         color="black", linestyle="--")
ax2.set_ylabel("Volume moyen annuel (log scale)")
ax2.set_yscale("log")

# Légendes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Polarisation économique vs Volume moyen du S&P 500")
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(12,6))

# Polarisation Fed et Monetary Policy
ax1.plot(df_all["year"], df_all["beta_fed"], label="Federal Reserve", color="tab:blue")
ax1.plot(df_all["year"], df_all["beta_monetary"], label="Monetary Policy", color="tab:red")
ax1.set_xlabel("Année")
ax1.set_ylabel("Polarisation (‖β‖₂)")

# Axe secondaire : croissance du volume en pointillé
ax2 = ax1.twinx()
ax2.plot(df_all["year"], df_all["sp500_volume_growth"], 
         label="Croissance du volume S&P 500", 
         color="black", linestyle="--", linewidth=2)
ax2.set_ylabel("Croissance annuelle du volume (log diff)")

# Légendes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Polarisation économique vs Croissance du volume du S&P 500")
plt.show()

In [ ]:
# Extraire juste les colonnes utiles (polarisation + croissance du volume)
subset = df_all[["beta_fed", "beta_monetary", "sp500_volume_growth"]]

# Corrélations de Pearson
print("Corrélations de Pearson :")
print(subset.corr(method="pearson"))

# Corrélations de Spearman
print("\nCorrélations de Spearman :")
print(subset.corr(method="spearman"))

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

# Préparer les données : supprimer les NaN et garder les deux colonnes
df_gc = df_all[["beta_monetary", "sp500_volume_growth"]].dropna()

# Test de Granger : est-ce que volume -> polarisation ?
# maxlag = nombre d'années de retard testées (ex. 3 ans)
print("Granger causality test : Volume -> Monetary Policy")
grangercausalitytests(df_gc[["beta_monetary", "sp500_volume_growth"]], maxlag=3)

# Inverse : polarisation -> volume
print("\nGranger causality test : Monetary Policy -> Volume")
grangercausalitytests(df_gc[["sp500_volume_growth", "beta_monetary"]], maxlag=3)
